# Exploratory Data Analysis v2 — Coffee Bean Quality Detection
### Sumber data: Cloudflare R2 (via DVC) — dijalankan di Kaggle Notebook
### Versi: FiftyOne

Notebook ini adalah **pasangan pembanding** dari `CBQD - EDA v2 (Manual).ipynb`. Cakupan
analisis dan urutan gap yang ditutup **sama persis** (lihat Daftar Isi), tapi beberapa
bagian dikerjakan dengan **FiftyOne** (dataset object, App interaktif, dan modul
**FiftyOne Brain**) supaya perbedaan & manfaatnya bisa dibandingkan langsung.

## Apa itu FiftyOne dan kenapa dipakai di sini?

FiftyOne adalah toolkit open-source untuk membangun, mengeksplorasi, dan meng-curate
dataset visual. Untuk audit EDA v1 yang dilakukan sebelumnya, ada 3 gap yang FiftyOne
bantu tangani lebih baik daripada kode pandas/OpenCV manual:

| Gap | Fitur FiftyOne yang dipakai |
|---|---|
| Leakage train↔test tidak pernah dicek | Dataset terpadu (train+test) + query lintas split |
| Sanity-check separability kelas | `fiftyone.brain.compute_visualization` (embedding scatter interaktif) |
| Audit label-noise | `fiftyone.brain.compute_mistakenness` (setelah baseline model) |
| (bonus) embedding CNN tanpa nulis training loop | `fiftyone.zoo` — load model pretrained siap pakai |

**Trade-off yang perlu disadari** (dibahas juga di Section 16 — Refleksi):
- FiftyOne + FiftyOne Brain menambah puluhan dependency (termasuk `fiftyone_db`, database
lokal berbasis MongoDB yang berjalan sebagai proses background).
- Nilai utamanya (App interaktif) tidak muncul kalau notebook ini dibaca tanpa dijalankan
live — setiap step berbasis App di sini selalu disertai fallback matplotlib statis.
- Beberapa API Brain yang lebih baru (mis. deteksi leaky-split, mistakenness) punya
signature yang bisa berubah antar versi; kode di bawah membungkusnya dengan try/except
dan fallback ke pendekatan manual yang sudah divalidasi supaya notebook tetap jalan
apa pun hasilnya.

**Prasyarat menjalankan di Kaggle:** Internet **On**, Secrets `R2_ACCESS_KEY_ID` /
`R2_SECRET_ACCESS_KEY` sudah diisi, `GIT_REPO_URL` di Section 2 sudah disesuaikan, dan
disarankan mengaktifkan GPU (Settings → Accelerator) supaya ekstraksi embedding CNN di
Section 10 lebih cepat (tetap jalan di CPU, hanya lebih lambat).

## Daftar Isi
1. Pendahuluan — Kenapa FiftyOne (di atas)
2. Environment & Data Provenance Setup
3. Membangun FiftyOne Dataset (Train + Test Terpadu)
4. Dataset Overview & Integrity Check
5. Class Distribution & Imbalance Analysis
6. Visual Sanity Check via FiftyOne App
7. Targeted Visual Stress Test (Field-Based Query)
8. Object-Level Shape, Size & Framing Analysis
9. Color Space, Intensity & Texture Distribution + Uji Statistik
10. Feature Embedding Extraction (FiftyOne Zoo)
11. Duplicate Detection & Train/Test Leakage Check
12. Near-Duplicate-Aware Split Strategy
13. Embedding Visualization & Baseline Sanity Check
14. Label-Noise / Mistakenness Audit
15. Preprocessing & Data Cleaning (Final Decision)
16. Kesimpulan Akhir & Refleksi: FiftyOne vs Manual

## Section 2 — Environment & Data Provenance Setup

Ini adalah bagian yang menggantikan `TRAIN_DIR = Path("/kaggle/input/...")` di v1.
Alih-alih menempel Kaggle Dataset, notebook ini menarik data dari **Cloudflare R2**
lewat **DVC**, mengikuti versi data yang sama dengan yang dipakai di pipeline project
(`dataset.dvc`, `dvc.yaml`, `metadata/manifest.csv`).

**Prasyarat di Kaggle:**
- Settings → Internet → **On**
- Kredensial R2 (`R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`) tersedia lewat salah satu dari:
  - **Private Kaggle Dataset** berisi `r2_credentials.json` (dipakai kalau run dipicu lewat
    `kaggle kernels push`/API — Kaggle Secrets yang di-attach lewat UI tidak ikut terbawa saat
    push via API/CLI, lihat [Kernel-Metadata wiki](https://github.com/Kaggle/kaggle-api/wiki/Kernel-Metadata)),
    attach dataset ini lewat Add-ons → Add Data di notebook editor atau `dataset_sources` di
    kernel-metadata.json; **atau**
  - **Kaggle Secrets** (Add-ons → Secrets) — otomatis dipakai sebagai fallback kalau file
    dataset di atas tidak ditemukan, cocok untuk run manual lewat tombol Save Version di UI.
- Kode project di-clone dari `https://github.com/Ardiyanto24/coffee-bean-quality-detection`
  (sudah diisi di `GIT_REPO_URL` di bawah — repo ini publik, berisi `dvc.yaml`, `dataset.dvc`,
  `.dvc/config`, `scripts/generate_manifest.py`, TANPA folder `dataset/` asli maupun kredensial
  R2 apa pun).

In [ ]:
# Sub-Step 2.1
# Tujuan: Install dependency tambahan: dvc[s3] untuk tarik data, fiftyone untuk EDA

!pip install -q "dvc[s3]" fiftyone "pillow<11"

In [ ]:
# Sub-Step 2.1b
# Tujuan: Cek kompatibilitas GPU SEBELUM torch pernah diimpor di proses ini

import os
import subprocess

# kernel-metadata.json cuma bisa minta "GPU nyala/mati" (enable_gpu), bukan tipe GPU
# tertentu -- Kaggle bisa saja mengalokasikan P100 (compute capability lama, sm_60)
# padahal wheel torch dari PyPI yang ditarik ulang oleh pip install fiftyone kerap
# hanya menyertakan kernel CUDA untuk arsitektur yang lebih baru (mis. T4/sm_75 ke atas),
# menyebabkan "CUDA error: no kernel image is available for execution on the device".
#
# Cek nama GPU lewat nvidia-smi (BUKAN lewat torch.cuda.*) sebelum torch pernah
# diimpor di proses ini sama sekali -- termasuk transitif lewat "import fiftyone".
# CUDA_VISIBLE_DEVICES hanya efektif jika di-set SEBELUM proses pertama kali
# menyentuh CUDA context; men-set-nya setelah torch sempat memanggil cuda.* (walau
# cuma get_device_capability) tidak akan mengubah apa pun lagi.
KNOWN_INCOMPATIBLE_GPUS = ("P100",)

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=15,
    )
    gpu_name = result.stdout.strip()
except Exception:
    gpu_name = ""

if gpu_name and any(bad in gpu_name for bad in KNOWN_INCOMPATIBLE_GPUS):
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    print(f"GPU '{gpu_name}' terdeteksi via nvidia-smi -- diketahui tidak kompatibel dengan "
          "build torch di sini, GPU dinonaktifkan sebelum torch pernah dipakai sama sekali.")
elif gpu_name:
    print(f"GPU '{gpu_name}' terdeteksi, dibiarkan aktif untuk dipakai torch/fiftyone.")
else:
    print("Tidak ada GPU terdeteksi (atau nvidia-smi tidak tersedia) -- jalan di CPU.")

In [ ]:
# Sub-Step 2.2
# Tujuan: Ambil source code project ke working directory Kaggle yang writable

import os

GIT_REPO_URL = "https://github.com/Ardiyanto24/coffee-bean-quality-detection.git"
PROJECT_DIR = "/kaggle/working/coffee-bean-quality-detection"

if GIT_REPO_URL:
    if not os.path.exists(PROJECT_DIR):
        os.system(f"git clone {GIT_REPO_URL} {PROJECT_DIR}")
else:
    # Alternatif: attach Kaggle Dataset berisi file-file DVC metadata (lihat catatan di atas),
    # lalu copy ke working dir karena /kaggle/input bersifat read-only dan dvc perlu menulis.
    KAGGLE_INPUT_DIR = "/kaggle/input/<nama-kaggle-dataset-dvc-meta>"
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.system(f"cp -r {KAGGLE_INPUT_DIR}/. {PROJECT_DIR}/")

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Sub-Step 2.3
# Tujuan: Konfigurasi kredensial R2 (private dataset jika ada, fallback ke Kaggle Secrets)

import json
from pathlib import Path

# kaggle kernels push (CLI/API) TIDAK membawa serta Kaggle Secrets yang di-attach lewat UI
# (keterbatasan resmi Kaggle: https://github.com/Kaggle/kaggle-api/wiki/Kernel-Metadata).
# Jalur utama: baca dari private Kaggle Dataset berisi r2_credentials.json.
# Fallback: Kaggle Secrets, dipakai otomatis saat notebook dijalankan manual dari UI (Save Version).
CREDENTIALS_DATASET_SLUG = "r2-credentials"  # ganti sesuai slug dataset Anda jika berbeda
cred_path = Path(f"/kaggle/input/{CREDENTIALS_DATASET_SLUG}/r2_credentials.json")

if cred_path.exists():
    creds = json.loads(cred_path.read_text())
    os.environ["AWS_ACCESS_KEY_ID"] = creds["R2_ACCESS_KEY_ID"]
    os.environ["AWS_SECRET_ACCESS_KEY"] = creds["R2_SECRET_ACCESS_KEY"]
    print("Kredensial R2 dimuat dari private Kaggle Dataset (nilai tidak di-print).")
else:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    os.environ["AWS_ACCESS_KEY_ID"] = secrets.get_secret("R2_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = secrets.get_secret("R2_SECRET_ACCESS_KEY")
    print("Kredensial R2 dimuat dari Kaggle Secrets (nilai tidak di-print).")

In [ ]:
# Sub-Step 2.4
# Tujuan: Tarik dataset dari R2 sesuai versi yang terekam di dataset.dvc

!dvc pull -v

In [ ]:
# Sub-Step 2.5
# Tujuan: Fallback: regenerate manifest bila belum ter-pull dari remote

from pathlib import Path

manifest_path = Path("metadata/manifest.csv")
if not manifest_path.exists():
    os.system("python scripts/generate_manifest.py")
print("Manifest tersedia:", manifest_path.exists())

In [ ]:
# Sub-Step 2.6
# Tujuan: Catat provenance versi data supaya kesimpulan EDA ini terikat ke versi dataset yang jelas

import yaml
import hashlib
import pandas as pd

with open("dataset.dvc") as f:
    dvc_meta = yaml.safe_load(f)
dataset_hash = dvc_meta["outs"][0]["md5"]
n_files_dvc = dvc_meta["outs"][0]["nfiles"]

manifest_df = pd.read_csv("metadata/manifest.csv")
manifest_hash = hashlib.md5(
    pd.util.hash_pandas_object(manifest_df, index=False).values.tobytes()
).hexdigest()

print(f"DVC dataset dir hash   : {dataset_hash}")
print(f"DVC dataset nfiles     : {n_files_dvc}")
print(f"Manifest rows          : {len(manifest_df)}")
print(f"Manifest content hash  : {manifest_hash}")
manifest_df.head()

**Kesimpulan Section 2** — Seluruh analisis di bawah ini terikat ke versi data dengan hash di atas. Jika dataset di-update di R2 (versi `dataset.dvc` baru), notebook ini harus dijalankan ulang; jangan bandingkan kesimpulannya dengan versi data yang berbeda tanpa mencatat hash-nya.

In [ ]:
# Sub-Step 2.7
# Tujuan: Setup path dasar yang dipakai di seluruh notebook (menggantikan re-scan folder manual di v1)

from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATASET_DIR = PROJECT_ROOT / "dataset"

manifest_df["abs_path"] = manifest_df["image_path"].apply(lambda p: str(DATASET_DIR / p))

train_df = manifest_df[manifest_df["split"] == "train"].reset_index(drop=True)
test_df = manifest_df[manifest_df["split"] == "test"].reset_index(drop=True)

class_names = sorted(train_df["label"].unique())
print("Classes  :", class_names)
print("N train  :", len(train_df))
print("N test   :", len(test_df))

## Section 3 — Membangun FiftyOne Dataset (Train + Test Terpadu)

Berbeda dari v1 yang scan folder manual berkali-kali, di sini train+test langsung disatukan menjadi **satu FiftyOne Dataset object** sejak awal — jadi query lintas split (Section 11) tinggal filter field `split`, tidak perlu join dataframe manual.

In [ ]:
# Sub-Step 3.1
# Tujuan: Load train sebagai FiftyOne Dataset dari struktur folder-per-kelas

import fiftyone as fo

DATASET_NAME = "coffee-bean-quality-v2"
if DATASET_NAME in fo.list_datasets():
    fo.delete_dataset(DATASET_NAME)

dataset = fo.Dataset.from_dir(
    dataset_dir=str(DATASET_DIR / "train"),
    dataset_type=fo.types.ImageClassificationDirectoryTree,
    name=DATASET_NAME,
)
dataset.persistent = True
print(dataset)

In [ ]:
# Sub-Step 3.2
# Tujuan: Tandai split=train, lalu tambahkan test set (tanpa label) sebagai sample terpisah

for sample in dataset.iter_samples(progress=True, autosave=True):
    sample["split"] = "train"

test_samples = [fo.Sample(filepath=str(p)) for p in test_df["abs_path"]]
for s in test_samples:
    s["split"] = "test"
dataset.add_samples(test_samples)

print(dataset.count_values("split"))

In [ ]:
# Sub-Step 3.3
# Tujuan: Hitung metadata gambar (width/height/num_channels/size) untuk seluruh sample

dataset.compute_metadata(progress=True)
print(dataset.first())

**Catatan struktur** — Section ini menggantikan `manifest_df`/`train_df`/`test_df` versi manual dengan satu object `dataset` (FiftyOne) yang menyimpan sample + field metadata sekaligus. Sepanjang notebook ini, `dataset.match(F("split") == "train")` dipakai sebagai pengganti `train_df`.

## Section 4 — Dataset Overview & Integrity Check

Setara Section 2 di Notebook Manual, memakai `dataset.exists()` dan `count_values()` alih-alih loop PIL manual.

In [ ]:
# Sub-Step 4.1
# Tujuan: Cek gambar yang gagal dibaca (tidak punya metadata = kemungkinan corrupt)

from fiftyone import ViewField as F

missing_meta_view = dataset.exists("metadata", False)
print(f"Kemungkinan corrupt/gagal dibaca: {len(missing_meta_view)} / {len(dataset)}")

In [ ]:
# Sub-Step 4.2
# Tujuan: Cek konsistensi channel warna & resolusi lewat metadata (v1 asumsikan RGB tanpa verifikasi)

print("num_channels:", dataset.count_values("metadata.num_channels"))
print("width       :", dataset.count_values("metadata.width"))
print("height      :", dataset.count_values("metadata.height"))
print("per split   :", dataset.count_values("split"))

**Kesimpulan Naratif — Section 4**

Sama seperti temuan Notebook Manual: pada versi dataset ini train (1.211) dan test (200) sama-sama 100% RGB (`num_channels=3`), resolusi seragam 256×256, tanpa sample yang gagal dibaca. `dataset.exists()`/`count_values()` mendapatkan kesimpulan yang sama dengan loop PIL manual, tapi lebih ringkas dan hasilnya langsung queryable lewat App di Section 6.

## Section 5 — Class Distribution & Imbalance Analysis

In [ ]:
# Sub-Step 5.1
# Tujuan: Distribusi kelas dari view train

train_view = dataset.match(F("split") == "train")
counts = train_view.count_values("ground_truth.label")

dist_df = pd.DataFrame(list(counts.items()), columns=["class_name", "num_images"])
dist_df["percentage"] = (dist_df["num_images"] / dist_df["num_images"].sum() * 100).round(2)
dist_df

In [ ]:
# Sub-Step 5.2
# Tujuan: Bar chart + rasio imbalance

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.bar(dist_df["class_name"], dist_df["num_images"])
plt.title("Class Distribution (Train Set)")
plt.tight_layout()
plt.show()

print("Imbalance ratio:", round(dist_df["num_images"].max() / dist_df["num_images"].min(), 3))

**Kesimpulan Naratif — Section 5** — Identik dengan Notebook Manual: distribusi kelas sangat seimbang (rasio ≈1.03), tidak diperlukan class weighting.

## Section 6 — Visual Sanity Check via FiftyOne App

Ini bagian yang paling terasa bedanya dibanding matplotlib grid statis: App FiftyOne memberi grid interaktif (klik gambar untuk detail, filter by field langsung dari UI). Karena App bersifat live/interaktif, setiap sel App di bawah disertai fallback matplotlib statis supaya notebook tetap informatif walau dibaca tanpa dijalankan.

In [ ]:
# Sub-Step 6.1
# Tujuan: Buka FiftyOne App HANYA saat edit interaktif (fo.launch_app bisa hang tanpa batas di run batch/API — lihat kesimpulan di bawah)

import os

# Kaggle set KAGGLE_KERNEL_RUN_TYPE="Interactive" saat notebook dibuka & dijalankan manual
# di editor, dan "Batch" saat dipicu via Save Version / kaggle kernels push (papermill).
# fo.launch_app() mencoba handshake ke front-end notebook untuk deteksi environment —
# di run Batch/papermill tidak ada front-end yang menjawab, sehingga panggilan ini bisa
# HANG TANPA BATAS (bukan error) dan menghabiskan kuota GPU sia-sia. Maka App hanya
# dibuka saat run terdeteksi interaktif; run batch otomatis pakai fallback matplotlib saja.
IS_INTERACTIVE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive") == "Interactive"

if IS_INTERACTIVE:
    session = fo.launch_app(dataset, auto=False)
    print("Buka App di:", session.url)
else:
    session = None
    print("Run batch terdeteksi (KAGGLE_KERNEL_RUN_TYPE != Interactive) — App dilewati, pakai fallback matplotlib.")

def set_app_view(view):
    """No-op saat run batch/App tidak dibuka, supaya sel-sel di bawah tetap aman dijalankan."""
    if session is not None:
        session.view = view

In [ ]:
# Sub-Step 6.2
# Tujuan: Arahkan App ke sample train (12 per kelas, seragam dengan sampling Notebook Manual)

SEED = 42
combined_ids = []
for cls in class_names:
    cls_view = train_view.match(F("ground_truth.label") == cls).take(12, seed=SEED)
    combined_ids.extend(s.id for s in cls_view)

set_app_view(dataset.select(combined_ids))
print(f"App menampilkan {len(combined_ids)} sample (12 per kelas)")

In [ ]:
# Sub-Step 6.3
# Tujuan: Fallback statis: grid matplotlib (identik dengan Section 4 Notebook Manual)

from PIL import Image

def plot_image_grid(image_paths, title, n_cols=4):
    n_images = len(image_paths)
    n_rows = (n_images + n_cols - 1) // n_cols
    plt.figure(figsize=(n_cols * 3, n_rows * 3))
    for i, p in enumerate(image_paths):
        plt.subplot(n_rows, n_cols, i + 1)
        plt.imshow(Image.open(p))
        plt.axis("off")
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

for cls in class_names:
    cls_view = train_view.match(F("ground_truth.label") == cls).take(12, seed=SEED)
    plot_image_grid([s.filepath for s in cls_view], title=f"Train — {cls}")

test_view_ = dataset.match(F("split") == "test").take(16, seed=SEED)
plot_image_grid([s.filepath for s in test_view_], title="Test set (unlabeled) — random sample")

**Kesimpulan Naratif — Section 6** — Sama seperti Notebook Manual: gambar test set konsisten secara visual dengan train, tidak ada domain-shift kentara. **Perbedaan workflow**: di FiftyOne, meninjau sample lain tinggal ganti `session.view` di UI (klik-klik) tanpa menulis ulang kode plotting — lebih cepat untuk eksplorasi eksploratif yang sifatnya iteratif/tidak terstruktur.

## Section 7 — Targeted Visual Stress Test (Field-Based Query)

Alih-alih sort manual dengan Python list, brightness dihitung sekali sebagai **field** lalu diquery dengan `sort_by` — hasilnya juga langsung bisa dibuka di App.

In [ ]:
# Sub-Step 7.1
# Tujuan: Hitung brightness sebagai field baru untuk seluruh sample

import numpy as np

for sample in dataset.iter_samples(progress=True, autosave=True):
    sample["brightness"] = float(np.array(Image.open(sample.filepath).convert("L")).mean())

print("Brightness range:", dataset.bounds("brightness"))

In [ ]:
# Sub-Step 7.2
# Tujuan: Ambil sample paling gelap & paling terang di kelas defect via sort_by

defect_view = train_view.match(F("ground_truth.label") == "defect")
dark_view = defect_view.sort_by("brightness")[:4]
bright_view = defect_view.sort_by("brightness", reverse=True)[:4]

extreme_ids = [s.id for s in dark_view] + [s.id for s in bright_view]
set_app_view(dataset.select(extreme_ids))
plot_image_grid([s.filepath for s in dark_view] + [s.filepath for s in bright_view],
                 title="Defect — Extreme Brightness (via field sort_by)")

**Kesimpulan Naratif — Section 7** — Fringing ungu/biru (chromatic aberration) di tepi bean tetap terlihat konsisten di semua kelas, seperti temuan di Notebook Manual — catatan yang sama berlaku untuk interpretasi edge density di Section 9.

## Section 8 — Object-Level Shape, Size & Framing Analysis *(Gap #3)*

Logika segmentasi **identik** dengan Notebook Manual (fungsi yang sama, sudah divalidasi terhadap data asli) — bedanya hasil disimpan sebagai field FiftyOne sehingga bisa langsung difilter/sort di App, bukan cuma di dataframe.

In [ ]:
# Sub-Step 8.1
# Tujuan: Fungsi segmentasi foreground vs background (sama seperti Notebook Manual)

import numpy as np
from PIL import Image

def foreground_stats(path):
    """Heuristik segmentasi background-putih vs bean (foreground lebih gelap).
    Mengembalikan area_frac, bbox_h, bbox_w, bbox_ratio, center_offset."""
    img = Image.open(path).convert("L")
    arr = np.array(img).astype(np.float32)
    thresh = arr.mean() - 0.6 * arr.std()
    mask = arr < thresh
    h, w = arr.shape
    if mask.sum() == 0:
        return None
    ys, xs = np.where(mask)
    area_frac = mask.sum() / (h * w)
    bbox_h = float(ys.max() - ys.min())
    bbox_w = float(xs.max() - xs.min())
    bbox_ratio = max(bbox_h, bbox_w) / max(min(bbox_h, bbox_w), 1e-6)
    cy, cx = ys.mean(), xs.mean()
    center_offset = float(np.hypot(cy - h / 2, cx - w / 2) / (h / 2))
    return {
        "area_frac": area_frac,
        "bbox_h": bbox_h,
        "bbox_w": bbox_w,
        "bbox_ratio": bbox_ratio,
        "center_offset": center_offset,
    }

In [ ]:
# Sub-Step 8.2
# Tujuan: Hitung & simpan sebagai field FiftyOne untuk seluruh sample

for sample in dataset.iter_samples(progress=True, autosave=True):
    stats_ = foreground_stats(sample.filepath)
    if stats_ is not None:
        sample["area_frac"] = stats_["area_frac"]
        sample["bbox_ratio"] = stats_["bbox_ratio"]
        sample["center_offset"] = stats_["center_offset"]

In [ ]:
# Sub-Step 8.3
# Tujuan: Export ke pandas untuk boxplot & ANOVA (dataset.values = pengganti groupby manual)

labels, area_fracs, bbox_ratios, center_offsets, splits = dataset.values(
    ["ground_truth.label", "area_frac", "bbox_ratio", "center_offset", "split"]
)
shape_df = pd.DataFrame({
    "label": labels, "area_frac": area_fracs, "bbox_ratio": bbox_ratios,
    "center_offset": center_offsets, "split": splits,
})
shape_df = shape_df[shape_df["split"] == "train"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
shape_df.boxplot(column="area_frac", by="label", ax=axes[0]); axes[0].set_title("area_frac")
shape_df.boxplot(column="bbox_ratio", by="label", ax=axes[1]); axes[1].set_title("bbox_ratio")
plt.suptitle(""); plt.tight_layout(); plt.show()

In [ ]:
# Sub-Step 8.4
# Tujuan: ANOVA (sama seperti Notebook Manual)

from scipy import stats

def anova_report(df, col, group_col="label"):
    groups = [g[col].dropna().values for _, g in df.groupby(group_col)]
    f, p = stats.f_oneway(*groups)
    df_between, df_within = len(groups) - 1, len(df) - len(groups)
    eta_sq = (f * df_between) / (f * df_between + df_within)
    return {"feature": col, "F": round(f, 3), "p_value": p, "eta_squared": round(eta_sq, 3)}

pd.DataFrame([anova_report(shape_df, c) for c in ["area_frac", "bbox_ratio", "center_offset"]])

**Kesimpulan Naratif — Section 8** — Angka yang sama persis dengan Notebook Manual (fungsi segmentasi identik): `longberry` paling elongated (bbox_ratio≈1.75), `premium` paling terpusat (center_offset≈0.18), semua perbedaan signifikan secara statistik dan cukup substansial (η²≈0.24–0.27). Nilai tambah FiftyOne di sini: field ini sekarang bisa langsung dipakai untuk filter di App, mis. `dataset.match(F("bbox_ratio") > 2.0)` untuk meninjau bean paling elongated secara visual satu per satu.

## Section 9 — Color, Intensity & Texture Distribution + Uji Statistik *(Gap #4)*

In [ ]:
# Sub-Step 9.1
# Tujuan: Hitung statistik warna & tekstur per-gambar sebagai field

import cv2

for sample in dataset.iter_samples(progress=True, autosave=True):
    arr = np.array(Image.open(sample.filepath).convert("RGB"))
    sample["mean_r"], sample["mean_g"], sample["mean_b"] = [float(arr[:, :, c].mean()) for c in range(3)]

    gray = cv2.cvtColor(cv2.imread(sample.filepath), cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    sample["edge_density"] = float(edges.mean() / 255)
    sample["variance"] = float(np.var(gray))

In [ ]:
# Sub-Step 9.2
# Tujuan: Export & uji statistik (ANOVA) per fitur

cols = ["ground_truth.label", "mean_r", "mean_g", "mean_b", "edge_density", "variance", "split"]
values = dataset.values(cols)
color_texture_df = pd.DataFrame(dict(zip(
    ["label", "mean_r", "mean_g", "mean_b", "edge_density", "variance", "split"], values
)))
color_texture_df = color_texture_df[color_texture_df["split"] == "train"]

pd.DataFrame([
    anova_report(color_texture_df, c)
    for c in ["mean_r", "mean_g", "mean_b", "edge_density", "variance"]
])

**Kesimpulan Naratif — Section 9** — Sama seperti Notebook Manual: warna (η²≈0.22) dan variance (η²≈0.22) adalah sinyal sedang-kuat begitu diukur per-gambar (bukan pool seluruh pixel seperti pendekatan histogram v1); edge_density signifikan tapi efeknya kecil (η²≈0.02). `premium` konsisten lebih gelap dan lebih tinggi variance-nya dibanding `peaberry`/`longberry`.

## Section 10 — Feature Embedding Extraction *(via FiftyOne Zoo)*

Ini salah satu keunggulan paling nyata FiftyOne dibanding Notebook Manual: Notebook Manual sengaja **menghindari** dependency CNN pretrained (pakai fitur hand-crafted saja) supaya tetap ringan. Di sini, `fiftyone.zoo` menyediakan model pretrained siap pakai untuk ekstraksi embedding **tanpa perlu menulis loop training/inference manual**.

In [ ]:
# Sub-Step 10.1
# Tujuan: Load model pretrained dari FiftyOne Model Zoo & hitung embeddings untuk seluruh dataset

import fiftyone.zoo as foz
import numpy as np
import torch

# Keputusan GPU/CPU sudah diambil di Sub-Step 2.1b lewat nvidia-smi, SEBELUM torch
# pernah diimpor di proses ini -- di sini torch tinggal melapor apa yang berlaku.
if torch.cuda.is_available():
    print(f"torch melihat GPU: {torch.cuda.get_device_name(0)}")
else:
    print("torch tidak melihat GPU (dinonaktifkan di Sub-Step 2.1b, atau memang tidak ada) -- jalan di CPU.")

model = foz.load_zoo_model("mobilenet-v2-imagenet-torch")
# skip_failures=False sengaja dipasang: default True di banyak versi FiftyOne akan
# menelan error per-sample secara diam-diam dan mengisi placeholder kosong/NaN, yang
# baru ketahuan berkeping-keping di section jauh setelah ini (mis. PCA/t-SNE error).
# num_workers=0 -- DataLoader multiprocessing dengan worker terpisah dikenal rawan
# deadlock di environment sandboxed/containerized seperti Kaggle; jalankan
# single-process saja (lebih lambat, tapi jauh lebih reliable).
embeddings = dataset.compute_embeddings(
    model, progress=True, skip_failures=False, num_workers=0
)
# Beberapa versi FiftyOne mengembalikan list of arrays alih-alih satu ndarray tersusun --
# np.asarray menyeragamkan keduanya jadi (N, D) tanpa peduli bentuk aslinya.
embeddings = np.asarray(embeddings)
print("Embeddings shape:", embeddings.shape)
assert embeddings.ndim == 2, (
    f"Embeddings harusnya 2D (N, D), didapat shape {embeddings.shape} -- "
    "kemungkinan compute_embeddings gagal diam-diam untuk sebagian/semua sample."
)

**Catatan** — Jika nama model `mobilenet-v2-imagenet-torch` tidak tersedia di versi FiftyOne Anda, jalankan `foz.list_zoo_models()` untuk melihat model pretrained lain yang tersedia (mis. varian ResNet/EfficientNet) dan ganti nama model di atas.

## Section 11 — Duplicate Detection & Train/Test Leakage Check *(Gap #1 & #2)*

Deteksi exact-duplicate & near-duplicate memakai **logika yang identik** dengan Notebook Manual (MD5 + pHash dengan threshold Hamming yang sama) supaya hasil kedua notebook bisa dibandingkan apple-to-apple — bedanya di sini hasilnya disimpan sebagai field/tag FiftyOne sehingga bisa langsung ditinjau di App. Sebagai bonus, disertakan juga cross-check berbasis embedding CNN (`fiftyone.brain.compute_similarity`) yang tidak punya padanan di Notebook Manual.

In [ ]:
# Sub-Step 11.1
# Tujuan: Exact duplicate (MD5) — simpan sebagai field & tag

import hashlib
from collections import Counter

def md5_file(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

for sample in dataset.iter_samples(progress=True, autosave=True):
    sample["md5"] = md5_file(sample.filepath)

md5_values = dataset.values("md5")
dup_md5 = {h for h, c in Counter(md5_values).items() if c > 1}

dup_view = dataset.match(F("md5").is_in(list(dup_md5)))
for sample in dup_view.iter_samples(autosave=True):
    sample.tags.append("exact_duplicate")

print(f"Exact-duplicate groups: {len(dup_md5)}  |  sample terlibat: {len(dup_view)}")

In [ ]:
# Sub-Step 11.2
# Tujuan: Near-duplicate & leakage via pHash (identik Notebook Manual)

import numpy as np
from PIL import Image
from scipy.fftpack import dct

def compute_phash(path, hash_size=8, highfreq_factor=4):
    """Reimplementasi pHash (ImageHash-style) tanpa dependency eksternal.
    Resize -> grayscale -> 2D DCT -> ambil blok frekuensi rendah -> threshold median."""
    img_size = hash_size * highfreq_factor
    img = Image.open(path).convert("L").resize((img_size, img_size), Image.LANCZOS)
    pixels = np.asarray(img, dtype=np.float64)
    d = dct(dct(pixels, axis=0), axis=1)
    low = d[:hash_size, :hash_size]
    med = np.median(low)
    return (low > med).flatten()

def hamming(a, b):
    return int(np.count_nonzero(a != b))

all_paths = dataset.values("filepath")
all_ids = dataset.values("id")
all_labels = dataset.values("ground_truth.label")
all_splits = dataset.values("split")

phash_matrix = np.stack([compute_phash(p) for p in all_paths])
packed = np.packbits(phash_matrix, axis=1)
prefix_keys = packed[:, 0].astype(np.uint16) * 256 + packed[:, 1].astype(np.uint16)

buckets = {}
for idx, key in enumerate(prefix_keys):
    buckets.setdefault(int(key), []).append(idx)

THRESH = 4
pairs = []
for idxs in buckets.values():
    if len(idxs) < 2:
        continue
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            a, b = idxs[i], idxs[j]
            dist = hamming(phash_matrix[a], phash_matrix[b])
            if dist <= THRESH:
                pairs.append((a, b, dist))

pairs_df = pd.DataFrame(pairs, columns=["idx_1", "idx_2", "hamming_dist"])
for name, arr in [("id", all_ids), ("label", all_labels), ("split", all_splits)]:
    pairs_df[f"{name}_1"] = pairs_df["idx_1"].map(lambda i: arr[i])
    pairs_df[f"{name}_2"] = pairs_df["idx_2"].map(lambda i: arr[i])

pairs_df["is_cross_split"] = pairs_df["split_1"] != pairs_df["split_2"]
same_split_train = (pairs_df["split_1"] == "train") & (pairs_df["split_2"] == "train")
pairs_df["is_cross_class"] = (pairs_df["label_1"] != pairs_df["label_2"]) & same_split_train

print(f"Total near-dup pairs: {len(pairs_df)}")
print(f"  cross-split (leakage): {pairs_df['is_cross_split'].sum()}")
print(f"  cross-class          : {pairs_df['is_cross_class'].sum()}")

In [ ]:
# Sub-Step 11.3
# Tujuan: Tag hasilnya di FiftyOne supaya bisa ditinjau di App

leak_ids = set(pairs_df.loc[pairs_df["is_cross_split"], "id_1"]) | set(pairs_df.loc[pairs_df["is_cross_split"], "id_2"])
cross_class_ids = set(pairs_df.loc[pairs_df["is_cross_class"], "id_1"]) | set(pairs_df.loc[pairs_df["is_cross_class"], "id_2"])

for sid in leak_ids:
    sample = dataset[sid]  # fetch sekali & simpan referensinya -- dataset[sid] dua kali
    sample.tags.append("cross_split_leak")  # bisa memicu ReferenceError (weakly-referenced
    sample.save()                            # object) karena instance pertama sempat di-GC
for sid in cross_class_ids:
    sample = dataset[sid]
    sample.tags.append("cross_class_near_dup")
    sample.save()

set_app_view(dataset.match_tags("cross_split_leak"))
print(f"Ditandai cross_split_leak: {len(leak_ids)} sample, cross_class_near_dup: {len(cross_class_ids)} sample")

In [ ]:
# Sub-Step 11.4
# Tujuan: (Bonus) Cross-check berbasis embedding CNN — fitur yang tidak ada di Notebook Manual

import fiftyone.brain as fob

try:
    fob.compute_similarity(dataset, embeddings=embeddings, brain_key="img_sim")
    test_view = dataset.match(F("split") == "test")
    train_view = dataset.match(F("split") == "train")

    nn_records = []
    for test_sample in test_view:
        res = train_view.sort_by_similarity(test_sample.id, brain_key="img_sim", k=1, dist_field="dist_to_query")
        nearest = res.first()
        nn_records.append({
            "test_path": test_sample.filepath,
            "nearest_train_path": nearest.filepath,
            "nearest_train_label": nearest.ground_truth.label,
            "distance": nearest["dist_to_query"],
        })
    embedding_nn_df = pd.DataFrame(nn_records).sort_values("distance")
    print("compute_similarity berhasil. 5 pasangan train<->test terdekat secara embedding:")
    print(embedding_nn_df.head())
except Exception as e:
    print(f"compute_similarity/sort_by_similarity tidak berjalan di versi ini ({e}).")
    print("Lewati bonus ini — hasil leakage utama tetap valid dari pHash di atas.")
    print("Tips: jalankan help(fob.compute_similarity) untuk cek signature versi Anda,")
    print("atau lihat https://docs.voxel51.com untuk contoh terbaru.")

**Kesimpulan Naratif — Section 11**

Hasil pHash (metode yang sama dengan Notebook Manual) menemukan **11 exact-duplicate group** (semua within-train) dan dari 65 near-duplicate pair: **7 leakage train↔test** dan **25 cross-class**. Nilai tambah FiftyOne di sini bukan pada angka (sama persis dengan Notebook Manual, sengaja dibuat identik untuk perbandingan yang adil), melainkan pada **cara meninjaunya**: `session.view = dataset.match(F("tags")...)` langsung membuka grid interaktif ke-7 pasangan leakage tadi di App, tanpa perlu menulis fungsi `show_pairs()` kustom seperti di Notebook Manual. Cross-check embedding CNN (jika berhasil) memberi perspektif semantik tambahan yang tidak dimiliki pHash (yang murni berbasis kemiripan piksel/frekuensi).

## Section 12 — Near-Duplicate-Aware Split Strategy *(Gap #2 — kelanjutan)*

Logika union-find + `StratifiedGroupKFold` identik dengan Notebook Manual; hasilnya disimpan sebagai field `cluster_id`/`fold` di FiftyOne sekaligus diekspor ke CSV.

In [ ]:
# Sub-Step 12.1
# Tujuan: Union-Find cluster dari near-duplicate DI DALAM TRAIN

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb

id_to_idx = {sid: i for i, sid in enumerate(all_ids)}
uf = UnionFind(len(all_ids))
train_pairs = pairs_df[(pairs_df["split_1"] == "train") & (pairs_df["split_2"] == "train")]
for _, r in train_pairs.iterrows():
    uf.union(id_to_idx[r["id_1"]], id_to_idx[r["id_2"]])

train_ids = [sid for sid, sp in zip(all_ids, all_splits) if sp == "train"]
root_to_cluster = {}
for sid in train_ids:
    root = uf.find(id_to_idx[sid])
    if root not in root_to_cluster:
        root_to_cluster[root] = len(root_to_cluster)
    sample = dataset[sid]  # simpan referensi -- dataset[sid] terpisah untuk set+save
    sample["cluster_id"] = root_to_cluster[root]  # memicu ReferenceError (weakly-referenced
    sample.save()                                  # object) karena instance pertama di-GC

print(f"Jumlah cluster unik: {len(root_to_cluster)} (dari {len(train_ids)} gambar train)")

In [ ]:
# Sub-Step 12.2
# Tujuan: StratifiedGroupKFold untuk assign fold, simpan sebagai field

from sklearn.model_selection import StratifiedGroupKFold

train_labels_ = np.array([dataset[sid].ground_truth.label for sid in train_ids])
train_clusters_ = np.array([dataset[sid]["cluster_id"] for sid in train_ids])

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
fold_assignment = np.full(len(train_ids), -1)
for fold, (_, val_idx) in enumerate(sgkf.split(train_ids, train_labels_, groups=train_clusters_)):
    fold_assignment[val_idx] = fold

for sid, fold in zip(train_ids, fold_assignment):
    sample = dataset[sid]
    sample["fold"] = int(fold)
    sample.save()

pd.Series(fold_assignment).value_counts().sort_index()

In [ ]:
# Sub-Step 12.3
# Tujuan: Sanity check + export manifest dengan fold

cluster_fold_df = pd.DataFrame({"cluster_id": train_clusters_, "fold": fold_assignment})
n_split_clusters = cluster_fold_df.groupby("cluster_id")["fold"].nunique().gt(1).sum()
assert n_split_clusters == 0, f"{n_split_clusters} cluster terbelah lintas fold!"

export_df = pd.DataFrame({
    "id": train_ids,
    "abs_path": [dataset[sid].filepath for sid in train_ids],
    "label": train_labels_,
    "cluster_id": train_clusters_,
    "fold": fold_assignment,
})
import os
os.makedirs("preprocessing_outputs", exist_ok=True)
export_df.to_csv("preprocessing_outputs/manifest_train_with_folds_fiftyone.csv", index=False)
print("OK — semua cluster utuh dalam satu fold. Manifest tersimpan.")

**Kesimpulan Naratif — Section 12** — Identik dengan Notebook Manual: 1.160 cluster unik dari 1.211 gambar train. Rekomendasi sama: pakai kolom `fold` ini untuk CV, jangan random split biasa.

## Section 13 — Embedding Visualization & Baseline Sanity Check *(Gap #5)*

Berbeda dari Notebook Manual yang memakai fitur hand-crafted, di sini dipakai **embedding CNN pretrained** dari Section 10 — baik untuk visualisasi maupun sebagai input baseline classifier, dengan CV yang tetap cluster-aware (fold dari Section 12).

In [ ]:
# Sub-Step 13.1
# Tujuan: 2D embedding visualization lewat FiftyOne Brain

import fiftyone.brain as fob

results = fob.compute_visualization(
    dataset, embeddings=embeddings, brain_key="img_viz", method="tsne", seed=51
)
if IS_INTERACTIVE:
    try:
        plot = results.visualize(labels="ground_truth.label")
        plot.show()
    except Exception as e:
        print(f"Interactive plot tidak tersedia di konteks ini ({e}); pakai fallback statis di bawah.")
else:
    print("Run batch terdeteksi — plot interaktif dilewati (bisa hang tanpa front-end), pakai fallback statis di bawah.")

In [ ]:
# Sub-Step 13.2
# Tujuan: Fallback statis: scatter matplotlib dari results.points

points = results.points
plot_labels = np.array(dataset.values("ground_truth.label"))
plot_labels = np.where(plot_labels == None, "test/unlabeled", plot_labels)

plt.figure(figsize=(7, 6))
for lbl in np.unique(plot_labels):
    mask = plot_labels == lbl
    plt.scatter(points[mask, 0], points[mask, 1], alpha=0.5, s=15, label=lbl)
plt.legend(); plt.title("t-SNE embedding CNN (MobileNetV2) — train + test")
plt.tight_layout(); plt.show()

In [ ]:
# Sub-Step 13.3
# Tujuan: Baseline classifier di atas embedding CNN, CV cluster-aware

from sklearn.model_selection import cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

split_arr = np.array(dataset.values("split"))
train_mask = split_arr == "train"
train_embeddings = embeddings[train_mask]

clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
y_pred = cross_val_predict(clf, train_embeddings, train_labels_, cv=sgkf, groups=train_clusters_)

print("CV accuracy (embedding CNN, cluster-aware):", round(accuracy_score(train_labels_, y_pred), 4))
print()
print(classification_report(train_labels_, y_pred))

In [ ]:
# Sub-Step 13.4
# Tujuan: Confusion matrix

import seaborn as sns

cm = confusion_matrix(train_labels_, y_pred, labels=class_names)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout(); plt.show()

**Kesimpulan Naratif — Section 13**

Bandingkan angka `CV accuracy` di atas dengan **≈72%** yang didapat Notebook Manual dari fitur hand-crafted. Embedding CNN pretrained (meski hanya untuk ImageNet, bukan dilatih khusus untuk bean) umumnya diharapkan menangkap tekstur & bentuk secara lebih kaya, sehingga akurasi baseline di sini biasanya lebih tinggi — jalankan kedua notebook secara langsung dan bandingkan angka pastinya di lingkungan Anda. Perhatikan juga apakah kelas yang paling sering tertukar (biasanya `defect`) tetap sama antara kedua pendekatan — jika iya, itu penguat kuat bahwa ambiguitas `defect` bukan artefak metode, melainkan karakter data itu sendiri.

## Section 14 — Label-Noise / Mistakenness Audit *(Gap #6)*

`fiftyone.brain.compute_mistakenness` adalah implementasi resmi untuk problem ini (dibanding heuristik manual `mistake_score` di Notebook Manual). Perlu prediksi model tersimpan sebagai `fo.Classification` di setiap sample.

In [ ]:
# Sub-Step 14.1
# Tujuan: Simpan prediksi out-of-fold sebagai field Classification

proba = cross_val_predict(clf, train_embeddings, train_labels_, cv=sgkf, groups=train_clusters_, method="predict_proba")
classes_ = clf.fit(train_embeddings, train_labels_).classes_

for sid, p in zip(train_ids, proba):
    pred_idx = int(np.argmax(p))
    sample = dataset[sid]
    sample["predictions"] = fo.Classification(
        label=str(classes_[pred_idx]),
        confidence=float(p[pred_idx]),
        logits=np.log(p + 1e-9).tolist(),
    )
    sample.save()

In [ ]:
# Sub-Step 14.2
# Tujuan: Jalankan compute_mistakenness (dengan fallback heuristik manual bila API berbeda)

try:
    fob.compute_mistakenness(
        dataset.match(F("split") == "train"), pred_field="predictions", label_field="ground_truth"
    )
    review_view = dataset.match(F("split") == "train").sort_by("mistakenness", reverse=True)
    n_flagged = len(review_view.match(F("mistakenness") > 0.5))
    print(f"compute_mistakenness berhasil. Kandidat review (mistakenness>0.5): {n_flagged}")
    set_app_view(review_view.limit(16))
except Exception as e:
    print(f"compute_mistakenness gagal ({e}); fallback ke heuristik manual (mistake_score).")
    true_proba = np.array([p[list(classes_).index(lbl)] for p, lbl in zip(proba, train_labels_)])
    mistake_score = proba.max(axis=1) - true_proba
    for sid, score in zip(train_ids, mistake_score):
        sample = dataset[sid]
        sample["mistake_score"] = float(score)
        sample.save()
    flagged_ids = [sid for sid, s in zip(train_ids, mistake_score) if s > 0.5]
    print(f"Kandidat review (mistake_score>0.5): {len(flagged_ids)}")
    set_app_view(dataset.select(flagged_ids))

**Kesimpulan Naratif — Section 14** — Baik lewat `compute_mistakenness` resmi maupun fallback manual, pola yang diharapkan konsisten dengan Notebook Manual: sekitar **~5% dari train** ditandai sebagai kandidat review, dengan `defect` mendominasi daftar tersebut. Keunggulan FiftyOne di sini: hasilnya langsung bisa ditinjau satu per satu di App (`session.view`) tanpa menulis fungsi grid kustom seperti di Notebook Manual.

## Section 15 — Preprocessing & Data Cleaning (Final Decision)

Menggabungkan tag yang sudah ditempel di Section 11 menjadi keputusan akhir data training, diekspor ke CSV yang sama formatnya dengan Notebook Manual.

In [ ]:
# Sub-Step 15.1
# Tujuan: Exclude exact_duplicate + cross_class_near_dup dari train, ekspor manifest bersih

exclude_view = dataset.match(F("split") == "train").match_tags(
    ["exact_duplicate", "cross_class_near_dup"]
)
exclude_ids = set(s.id for s in exclude_view)

clean_train_ids = [sid for sid in train_ids if sid not in exclude_ids]
print(f"Train sebelum: {len(train_ids)}  ->  sesudah cleaning: {len(clean_train_ids)}")

clean_export_df = export_df[export_df["id"].isin(clean_train_ids)]
clean_export_df.to_csv("preprocessing_outputs/manifest_train_clean_fiftyone.csv", index=False)

In [ ]:
# Sub-Step 15.2
# Tujuan: Tandai (bukan hapus) test yang match cross_split_leak, ekspor

test_flag_df = pd.DataFrame({
    "id": dataset.match(F("split") == "test").values("id"),
    "abs_path": dataset.match(F("split") == "test").values("filepath"),
})
leak_test_ids = set(s.id for s in dataset.match(F("split") == "test").match_tags("cross_split_leak"))
test_flag_df["possible_train_leak"] = test_flag_df["id"].isin(leak_test_ids)
test_flag_df.to_csv("preprocessing_outputs/manifest_test_flagged_fiftyone.csv", index=False)

print(f"Test ditandai possible_train_leak: {test_flag_df['possible_train_leak'].sum()} / {len(test_flag_df)}")

**Kesimpulan Naratif — Section 15** — Hasil akhir (jumlah file dikecualikan, jumlah test yang ditandai) identik dengan Notebook Manual karena sumber tag/kriterianya sama. Bedanya proses ini bisa dilakukan interaktif: sebelum export, tag exclude bisa ditinjau & di-adjust manual dulu di App (mis. batalkan tag pada pasangan yang setelah dilihat langsung ternyata bukan duplikat).

## Section 16 — Kesimpulan Akhir & Refleksi: FiftyOne vs Manual

**Temuan inti sama persis dengan Notebook Manual** (memang didesain agar sebanding): dataset seimbang & bersih secara integritas, bentuk/warna/variance adalah sinyal morfologi nyata, ada 7 leakage train↔test dan 32 near-dup dalam-train yang perlu cluster-aware split, dan ±5% train perlu ditinjau untuk kemungkinan mislabel.

**Refleksi FiftyOne vs Manual (pandas/OpenCV/scikit-learn):**

| Aspek | Manual | FiftyOne |
|---|---|---|
| Setup / dependency | Ringan, semua sudah lazim terpasang | Berat: ~40 package + local MongoDB (`fiftyone_db`) |
| Visual sanity check | Statis (matplotlib grid), perlu tulis ulang kode tiap ganti sample | Interaktif (App), ganti tampilan tanpa tulis ulang kode |
| Duplicate/leakage | Kode pHash manual, hasil di dataframe | Logika sama + hasil langsung *taggable* & *browsable* di App |
| Embedding CNN | Sengaja dihindari (supaya ringan) — pakai fitur hand-crafted | `fiftyone.zoo` menyediakan model pretrained dalam 1 baris kode |
| Embedding visualization | PCA manual dengan sklearn | `compute_visualization` + App interaktif (klik titik -> lihat gambar) |
| Label-noise audit | Heuristik `mistake_score` buatan sendiri | `compute_mistakenness` — implementasi resmi, lebih teruji |
| Portabilitas notebook | Jalan di hampir semua environment Python | Butuh environment yang mendukung MongoDB lokal (Kaggle OK, environment terbatas mungkin bermasalah) |
| Reproducibility API | Stabil (pandas/sklearn API jarang berubah drastis) | Beberapa API Brain lebih baru, signature bisa berubah antar versi (perlu try/except seperti di notebook ini) |

**Kapan pakai yang mana:** untuk EDA cepat/ringan atau environment terbatas (mis. CI, server tanpa Docker/MongoDB), Notebook Manual lebih portable. Untuk eksplorasi iteratif yang butuh banyak visual inspection bolak-balik (curation dataset, review manual label error, cek kualitas anotasi), FiftyOne App memangkas waktu iterasi secara signifikan — trade-off dependency-nya sepadan begitu ukuran dataset/tim bertambah besar.